In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prd","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)
goldTablName = f"saleslake_{env}.gold_{env}.refinedcustomer"
# print(goldTablName)
silverTablName = f"saleslake_{env}.silver_{env}.cleanedcustomer"

In [0]:
result_gold = spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (

    WITH latest_customer_silver AS (
        SELECT *
        FROM {silverTablName}
    ),

    latest_rm_dup_silver AS (
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY customer_id
                       ORDER BY ingest_ts DESC
                   ) AS rn
            FROM latest_customer_silver
        )
        WHERE rn = 1
    ),

    silver_gold_rec AS (
        SELECT
            s.*,
            g.is_active,
            g.rec_version,
            g.start_effective_ts,
            g.end_effective_ts,

            CASE
                WHEN g.customer_id IS NULL THEN 1
                ELSE g.rec_version + 1
            END AS new_rec_version,

            CASE
                WHEN g.customer_id IS NULL THEN 'NEW'

                WHEN NOT (s.customer_name <=> g.customer_name)
                  OR NOT (s.email         <=> g.email)
                  OR NOT (s.phone         <=> g.phone)
                  OR NOT (s.address       <=> g.address)
                  OR NOT (s.city          <=> g.city)
                  OR NOT (s.state         <=> g.state)
                  OR NOT (s.country       <=> g.country)
                  OR NOT (s.zip_code      <=> g.zip_code)
                  OR NOT (s.segment       <=> g.segment)

                THEN 'CHANGE'

                ELSE 'NO_CHANGE'
            END AS rec_flag

        FROM latest_rm_dup_silver s

        LEFT JOIN (
            SELECT *
            FROM {goldTablName}
            WHERE is_active = 'Y'
        ) g
        ON s.customer_id = g.customer_id
    ),

    insert_flag AS (
        SELECT
            NULL AS cust_merge_key,

            customer_id,
            customer_name,
            email,
            phone,
            address,
            city,
            state,
            country,
            zip_code,
            segment,

            new_rec_version,
            rec_flag,

            'INSERT' AS merge_flag

        FROM silver_gold_rec
        WHERE rec_flag IN ('NEW','CHANGE')
    ),

    changed_flag AS (
        SELECT
            customer_id AS cust_merge_key,

            customer_id,
            customer_name,
            email,
            phone,
            address,
            city,
            state,
            country,
            zip_code,
            segment,

            new_rec_version,
            rec_flag,

            'UPDATE' AS merge_flag

        FROM silver_gold_rec
        WHERE rec_flag = 'CHANGE'
    )

    SELECT * FROM changed_flag

    UNION ALL

    SELECT * FROM insert_flag

) src

ON tgt.customer_id = src.cust_merge_key
AND tgt.is_active = 'Y'

WHEN MATCHED
AND src.merge_flag = 'UPDATE'
THEN UPDATE SET
    tgt.is_active = 'N',
    tgt.end_effective_ts = current_timestamp()

WHEN NOT MATCHED
AND src.merge_flag = 'INSERT'
THEN INSERT (
    customer_id,
    customer_name,
    email,
    phone,
    address,
    city,
    state,
    country,
    zip_code,
    segment,
    is_active,
    rec_version,
    start_effective_ts,
    end_effective_ts
)
VALUES (
    src.customer_id,
    src.customer_name,
    src.email,
    src.phone,
    src.address,
    src.city,
    src.state,
    src.country,
    src.zip_code,
    src.segment,
    'Y',
    src.new_rec_version,
    current_timestamp(),
    TO_TIMESTAMP('9999-12-31','yyyy-MM-dd')
)
""")

In [0]:
display(result_gold)